# arXiv Crawler — Colab (Milestone 1)

Notebook này hiển thị tiến trình crawl theo thời gian thực: **discovery**, **metadata**, **source** và **references theo version**.

## Cách chạy

1. Chạy lần lượt các cell thiết lập phía trên.
2. Chạy cell **Status** để xem dữ liệu đã crawl.
3. Chạy **một** cell chế độ crawl (Smoke / Full / arXiv-only / S2-only). Không dùng Run all.

`PROGRESS_EVERY_SECONDS` điều khiển nhịp in tiến trình (mặc định 30 giây). Các mốc startup/finalize vẫn in ngay.

## Quy tắc quan trọng

- Không xoá `progress/`, `jobs.journal.jsonl`, cooldown.
- Giữ nguyên `persistent_root` và `range` khi resume cùng dataset.
- Failed jobs **không** tự retry khi resume.
- Chỉ bật `TAKEOVER_STALE=True` khi chắc chắn runtime cũ đã dừng.


## 1. Mount Drive và chọn thư mục dự án

Sửa `PROJECT_DIR` nếu thư mục dự án trên Drive của bạn khác.


In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT_DIR = '/content/drive/MyDrive/arxiv'

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    print('Không phải Colab: bỏ qua mount Drive')

root = Path(PROJECT_DIR).expanduser()
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)
print('Project root:', root)


## 2. Lấy/cập nhật code

Cell này fetch `main` từ repo và `reset --hard`, nên mọi sửa đổi local trong `code/` sẽ bị ghi đè. Range mong muốn đã được khai báo trong `config/crawler.yaml` của repo.


In [ ]:
repo_url = 'https://github.com/KienBuiTan-ichica-1/arxiv-kien.git'
repo = Path('code')
if not (repo / '.git').exists():
    if repo.exists():
        import shutil
        shutil.rmtree(repo)
    subprocess.run(['git', 'clone', repo_url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'remote', 'set-url', 'origin', repo_url], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'src/requirements.txt')], check=True)
sys.path.insert(0, str(repo / 'src'))
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'log', '-1', '--oneline'], text=True).strip())


## 3. Nạp cấu hình và kiểm tra key

Cell in range, persistent root, commit và thông số S2. Thiếu key chỉ cảnh báo; cell chạy crawl sẽ báo lỗi rõ ràng.


In [ ]:
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key
from crawler.ids import generate_ids
import traceback
from crawler.console import log

config = load_config(Path('code/config/crawler.yaml'))
config.execution['allow_anonymous_s2'] = False

def require_s2_key():
    if not load_semantic_scholar_key():
        raise RuntimeError('Thiếu SEMANTIC_SCHOLAR_API_KEY. Thêm trong Colab Secrets (biểu tượng chìa khóa), bật Notebook access, rồi chạy lại cell này.')

async def run_selected(stages, limit, takeover_stale=False):
    log('Notebook: run_selected entered; checking required S2 key')
    require_s2_key()  # Bắt buộc cả khi chỉ chạy arXiv; không in key.
    log('Notebook: S2 key present; entering pipeline')
    config.execution['allow_anonymous_s2'] = False
    try:
        result = await run_pipeline(config, stages, resume=True, limit=limit,
                                    takeover_stale=takeover_stale)
    except BaseException:
        traceback.print_exc()
        raise
    print(result)
    return result

input_ids = generate_ids(**config.range)
print('Range inclusive:', input_ids[0], '→', input_ids[-1], '| Tổng:', len(input_ids))
print('Persistent root (absolute):', Path(config.paths['persistent_root']).resolve())
print('Scratch root:', config.paths['scratch_root'])
print('Commit:', subprocess.check_output(['git', '-C', 'code', 'log', '-1', '--oneline'], text=True).strip())
if load_semantic_scholar_key():
    print('S2 key: đã tìm thấy (chưa kiểm chứng quyền truy cập API)')
else:
    print('CẢNH BÁO: chưa tìm thấy SEMANTIC_SCHOLAR_API_KEY; cell chạy crawl sẽ lỗi cho tới khi bổ sung.')
print('Reference query: URL arXiv vN; chưa kiểm chứng bibliography lịch sử')
print('S2 gap (giây):', config.network['s2_key_gap_seconds'])
print('S2 batch size:', config.network['s2_batch_size'])
print('Publish queue S2:', config.execution['s2_publish_queue_capacity'], 'batch/page /', config.execution['s2_publish_queue_bytes'], 'bytes')


## 4. Hiển thị tiến trình

Cell này định nghĩa dashboard tiến trình và hàm chạy crawl có in tiến trình định kỳ.

- `PROGRESS_EVERY_SECONDS`: số giây giữa hai lần in (đặt nhỏ hơn nếu muốn cập nhật dày hơn).
- Tiến trình được tính **trong phạm vi ID đang chạy** (smoke = 10 ID, full = toàn range).
- Mỗi stage hiển thị `hoàn tất / tổng (phần trăm)` kèm chi tiết trạng thái.


In [ ]:
import asyncio, time
from collections import Counter
from crawler.progress import ProgressStore

PROGRESS_EVERY_SECONDS = 30

STAGES = ('discovery', 'metadata', 'source', 'references')
TERMINAL = ('succeeded', 'no_tex', 'not_found', 'failed')
DETAIL_KEYS = ('succeeded', 'not_found', 'no_tex', 'failed', 'running', 'pending')

def _elapsed_text(seconds):
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

def _scope_rows(store, selected_ids=None):
    scoped = {}
    for key, row in store.rows.items():
        if selected_ids is not None and key[0] not in selected_ids:
            continue
        stage = key[2]
        if stage == 'references':
            if key[3] != 0:
                continue
            if key[1] == 0:
                stage = 'legacy_references'
        scoped.setdefault(stage, []).append(row)
    return scoped

def crawl_progress_lines(store, selected_ids=None):
    scoped = _scope_rows(store, selected_ids)
    lines = []
    for stage in STAGES:
        rows = scoped.get(stage, [])
        if not rows:
            continue
        total = len(rows)
        counts = Counter(r['status'] for r in rows)
        terminal = sum(counts.get(s, 0) for s in TERMINAL)
        pct = (terminal / total * 100) if total else 0
        detail = ' '.join(f'{s}={counts.get(s, 0)}' for s in DETAIL_KEYS)
        lines.append(f'{stage:<11} {terminal:>5}/{total:<5} ({pct:5.1f}%)  {detail}')
    legacy = scoped.get('legacy_references', [])
    if legacy:
        counts = Counter(r['status'] for r in legacy)
        detail = ' '.join(f'{k}={v}' for k, v in sorted(counts.items()))
        lines.append(f'legacy_refs {len(legacy):>5}{"":<6}          {detail}')
    return lines

def print_crawl_progress(total_papers=None, started=None, selected_ids=None):
    try:
        store = ProgressStore(config.paths['persistent_root'])
    except Exception as exc:
        print(f'[progress] chưa đọc được: {type(exc).__name__}: {exc}', flush=True)
        return None
    clock = time.strftime('%H:%M:%S')
    elapsed = f' | đã {_elapsed_text(time.time() - started)}' if started else ''
    print(f'[{clock}]{elapsed}', flush=True)
    for line in crawl_progress_lines(store, selected_ids):
        print('    ' + line, flush=True)
    return store

async def run_with_progress(stages, limit, takeover_stale=False):
    selected_ids = set(input_ids[:limit]) if limit else set(input_ids)
    label = f'smoke {len(selected_ids)} ID' if limit else 'toàn range'
    started = time.time()
    print(f'=== Bắt đầu {label} | stages={sorted(stages)} | takeover={takeover_stale} ===', flush=True)
    print(f'    Cập nhật mỗi {PROGRESS_EVERY_SECONDS} giây', flush=True)
    task = asyncio.create_task(run_selected(stages, limit, takeover_stale=takeover_stale))
    try:
        while not task.done():
            await asyncio.to_thread(print_crawl_progress, None, started, selected_ids)
            await asyncio.wait({task}, timeout=PROGRESS_EVERY_SECONDS)
        summary = await task
    except BaseException:
        print('Đang huỷ pipeline và chờ lưu progress...', flush=True)
        task.cancel()
        try:
            await task
        except BaseException:
            pass
        raise
    print(f'=== Hoàn tất sau {_elapsed_text(time.time() - started)} ===', flush=True)
    print_crawl_progress(None, started, selected_ids)
    return summary


## 5. Scratch SSD tùy chọn

Chạy sau cell cấu hình để tải/giải nén trên SSD; result và progress vẫn lưu tại `persistent_root` trên Drive. Symlink hỏng sau runtime reset được tạo lại.


In [ ]:
USE_LOCAL_SCRATCH = True
if USE_LOCAL_SCRATCH:
    import tempfile
    scratch_dir = Path('scratch')
    if scratch_dir.is_symlink() and not scratch_dir.exists():
        scratch_dir.unlink()
    if not scratch_dir.exists():
        scratch_dir.symlink_to(tempfile.mkdtemp(prefix='arxiv-crawler-'), target_is_directory=True)
    assert scratch_dir.is_dir()
    config.paths['scratch_root'] = 'scratch'
    config.validate()
print('Scratch project path:', Path(config.paths['scratch_root']).resolve())


## 6. Status — xem dữ liệu đã crawl

Chỉ đọc dữ liệu đã lưu, không gọi API, không reset job/cooldown. Chạy sau khi cell crawler dừng hẳn để có snapshot ổn định.

`references` đếm **version snapshots**; `legacy_references` là snapshot v0 cũ. Không cộng hai số này.


In [ ]:
from crawler.progress import ProgressStore
from datetime import datetime, timezone
from collections import Counter
import json, time

root = Path(config.paths['persistent_root'])
progress = root / 'progress'
if not (progress / 'jobs.csv').exists() and not (progress / 'jobs.bak').exists():
    print('Chưa có snapshot progress tại:', progress)
else:
    store = ProgressStore(root)
    ids = set(input_ids)
    print('Range:', input_ids[0], '→', input_ids[-1], '| papers:', len(input_ids))
    print('\n--- Trong range hiện tại ---')
    for line in crawl_progress_lines(store, ids):
        print('    ' + line)
    print('\n--- Toàn bộ ledger đã lưu ---')
    for line in crawl_progress_lines(store, None):
        print('    ' + line)

    version_refs = [r for r in store.rows.values()
                    if r['stage'] == 'references' and r['version'] > 0 and r['offset'] == 0]
    active_refs = [r for r in version_refs if r['status'] in ('pending', 'running')]
    batch_progress = Counter((r['reference_pass'], r['batch_cap']) for r in active_refs if r['reference_pass'] <= 4)
    print('\nS2 batch pending/running theo (pass, cap):', dict(sorted(batch_progress.items())))
    print('S2 pagination pass >= 5 pending/running:', sum(r['reference_pass'] >= 5 for r in active_refs))

    failures = [r for r in store.rows.values() if r['status'] == 'failed']
    print('Job failed:', len(failures), '(resume không tự retry)')
    for row in failures[:20]:
        print('   ', {k: row[k] for k in ('paper', 'version', 'stage', 'offset', 'attempts', 'error')})
    if len(failures) > 20:
        print('    ... chỉ hiển thị 20 lỗi đầu')

services = progress / 'services.json'
if services.exists():
    for service, state in json.loads(services.read_text()).items():
        deadline = state.get('deadline', 0)
        print(service, 'cooldown đến:', datetime.fromtimestamp(deadline, timezone.utc).isoformat(),
              '| còn chờ (giây):', round(max(0, deadline - time.time())),
              '| backoff:', state.get('backoff', 0))
print('Status chỉ đọc dữ liệu đã lưu; Drive có thể đồng bộ chậm.')


## 7. Smoke — tối đa 10 ID đầu

Đổi `RUN_SMOKE=True` rồi chạy. Dữ liệu thành công được bỏ qua khi resume; failed jobs không tự retry.


In [ ]:
RUN_SMOKE = False
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_SMOKE:
    summary = await run_with_progress({'arxiv', 's2'}, 10, takeover_stale=TAKEOVER_STALE)
else:
    print('Chưa chạy smoke: đặt RUN_SMOKE=True khi sẵn sàng.')


## 8. Toàn range — arXiv + S2

Chạy discovery, metadata, source và references theo version cho toàn range trong config. Giữ đúng `persistent_root` và range để resume.


In [ ]:
RUN_FULL_RANGE = False  # Đổi True để chủ động chạy chế độ này.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_FULL_RANGE:
    summary = await run_with_progress({'arxiv', 's2'}, None, takeover_stale=TAKEOVER_STALE)
else:
    print('Bỏ qua: RUN_FULL_RANGE=False')


## 9. Chỉ arXiv — toàn range

Chạy discovery, metadata và source, không gửi request S2. Dùng khi S2 đang cooldown.


In [ ]:
RUN_ARXIV_ONLY = False  # Đổi True để chủ động chạy chế độ này.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_ARXIV_ONLY:
    summary = await run_with_progress({'arxiv'}, None, takeover_stale=TAKEOVER_STALE)
else:
    print('Bỏ qua: RUN_ARXIV_ONLY=False')


## 10. Chỉ Semantic Scholar — toàn range

Chạy references cho các version đã có trong ledger. Nếu chưa discovery đủ, chạy arXiv trước.

`S2_LIMIT=10` để thử 10 paper đầu; `None` là toàn range.


In [ ]:
RUN_S2_ONLY = False  # Đổi True để chủ động chạy chế độ này.
S2_LIMIT = None  # Đặt 10 để thử riêng S2 cho 10 paper đầu; None = toàn range.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_S2_ONLY:
    summary = await run_with_progress({'s2'}, S2_LIMIT, takeover_stale=TAKEOVER_STALE)
else:
    print('Bỏ qua: RUN_S2_ONLY=False')


## 11. Chẩn đoán source (tùy chọn)

Chỉ xem chẩn đoán, không reset failed.


In [ ]:
from crawler.diagnostics import diagnose_sources
import json
print(json.dumps(diagnose_sources(config), indent=2, ensure_ascii=False))


## 12. Export snapshot (tùy chọn)

Chỉ chạy sau khi crawler đã dừng. Export gồm `version_references.parquet` và `reference_snapshots.parquet`; `references.parquet` là union cấp paper.


In [ ]:
EXPORT_SNAPSHOT = False
if EXPORT_SNAPSHOT:
    from crawler.export import export_snapshot
    import uuid
    print(export_snapshot(config, 'export-' + uuid.uuid4().hex))
else:
    print('Bỏ qua: EXPORT_SNAPSHOT=False')


## 13. Audit checksum (tùy chọn)

Chạy khi crawler đã dừng. Audit không gọi API, không tự sửa/reset job.


In [ ]:
RUN_AUDIT = False
FULL_AUDIT = False
TAKEOVER_STALE = False
if RUN_AUDIT:
    from crawler.verification import audit
    audit_summary = audit(config, ids=set(generate_ids(**config.range)),
                          full=FULL_AUDIT, takeover_stale=TAKEOVER_STALE)
    print(audit_summary)
else:
    print('Bỏ qua: RUN_AUDIT=False')


## 14. Đo tổng dung lượng (tùy chọn)

Quét toàn cây có thể rất lâu trên Drive. Không cần chạy để resume crawler.


In [ ]:
MEASURE_TOTAL_DISK = False
TAKEOVER_STALE = False
if MEASURE_TOTAL_DISK:
    from crawler.disk_usage import DiskUsage
    from crawler.locking import DatasetLock
    import uuid
    with DatasetLock(config.paths['persistent_root'], 'disk-' + uuid.uuid4().hex, TAKEOVER_STALE):
        print('Bắt đầu quét toàn bộ dung lượng; Drive có thể rất chậm.', flush=True)
        total_bytes = DiskUsage(config.paths.values()).total
        print('Tổng logical bytes trong persistent/scratch roots:', total_bytes, flush=True)
else:
    print('Bỏ qua: MEASURE_TOTAL_DISK=False')


## Ghi chú vận hành

### Dữ liệu nằm ở đâu
- Kết quả crawl: `runtime/data/<arxiv_id>/`
- Progress: `runtime/progress/` (`jobs.csv`, `jobs.journal.jsonl`, `papers.csv`, `versions.csv`)
- Export: `runtime/exports/<run_id>/`
- `scratch/` chỉ là vùng tạm trên SSD, bị xoá sau publish.

### Lỗi “Previous active marker exists”
Nghĩa là lần chạy trước chưa thoát sạch, còn `runtime/progress/active_run.json`. Chỉ bật `TAKEOVER_STALE = True` sau khi đã restart/terminate runtime cũ và chắc chắn không còn process nào ghi dataset; chạy một lần rồi đổi lại `False`.

### Resume và failed jobs
- Resume bỏ qua job đã `succeeded`; không tải/upload lại.
- Job `pending`/`running` dở được chạy tiếp.
- Job `failed` không tự retry; cần xử lý riêng.
- Không xoá journal, cooldown hoặc đổi `persistent_root` khi đang resume cùng dataset.

### Đổi range
Sửa `config/crawler.yaml` trong repo (trên Drive là `code/config/crawler.yaml`), rồi chạy lại cell cấu hình. Range xuyên tháng cần `monthly_end_ids` cho tháng đầu và các tháng trung gian.
